# Assignment 5 — Part B1: Loan Data Processing

**Dataset:** `raw_loan_dataset.csv`  
**Goal:** Clean, normalize, and engineer features from the raw loan dataset and save the result as `clean_loan_dataset.csv` for use in Part B2.  
**Steps:** Load → Clean Currency → Fix Categories → Impute → Duplicates → Outliers → Encode → Class Balance → Feature Engineering → Scale → Save

## Step 1 — Load & Inspect

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('raw_loan_dataset.csv')

print('=== HEAD ===')
print(df.head(10))
print()
print('=== INFO ===')
df.info()
print()
print('=== MISSING VALUE COUNTS ===')
print(df.isnull().sum())
print(f'\nDataset shape: {df.shape}')

=== HEAD ===
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   
5      NaN        754.0              3.6      47800           Yes   
6   106070        665.0             14.6      48000             N   
7    35458        599.0             21.7    $10,300            No   
8    73520        661.0              NaN      17200           Yes   
9    57087        563.0             11.8      13400           Yes   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  
5               No       Yes  
6         

**Key issues found:**
- `Income` and `LoanAmount` contain currency-formatted strings (e.g. `"$25,851"`) instead of plain numbers.
- `HasCollateral` has mixed spellings: `Yes`, `Y`, `yes`, `yse`, `N`, `No`.
- `PreviousDefaults` has mixed spellings: `No`, `Yes`, `Y`, `N`, `1`, `0`.
- `Approved` has variants: `Yes`, `No`, `approved`, `rejected`, `Approved`, `Rejected`, `YES`, `NO`.
- Several numeric columns have missing values: `Income`, `CreditScore`, `EmploymentYears`, `LoanAmount`, `HasCollateral`, `PreviousDefaults`.

## Step 2 — Clean Currency Formatting

In [2]:
# Remove '$' and ',' from Income and LoanAmount and convert to float
df['Income'] = df['Income'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['Income'] = pd.to_numeric(df['Income'], errors='coerce')

df['LoanAmount'] = df['LoanAmount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['LoanAmount'] = pd.to_numeric(df['LoanAmount'], errors='coerce')

print('CHECKPOINT — Currency cleaned.')
print(df[['Income', 'LoanAmount']].describe())

CHECKPOINT — Currency cleaned.
              Income     LoanAmount
count      98.000000      98.000000
mean    73376.234694   26633.673469
std     33746.896559   21303.559278
min     25851.000000    5000.000000
25%     45840.750000   13575.000000
50%     69460.500000   20700.000000
75%     96482.000000   35175.000000
max    250000.000000  180000.000000


## Step 3 — Fix Category Errors (Before Imputation)

In [3]:
# --- HasCollateral ---
# Normalize to Yes / No
collateral_map = {
    'yes': 'Yes', 'y': 'Yes', 'yse': 'Yes',  # typos / abbreviations for Yes
    'no': 'No', 'n': 'No'                     # abbreviations for No
}
df['HasCollateral'] = df['HasCollateral'].astype(str).str.strip().str.lower().map(
    lambda x: collateral_map.get(x, x.capitalize() if x not in ('nan',) else np.nan)
)
df['HasCollateral'] = df['HasCollateral'].replace('Nan', np.nan)
print('HasCollateral value_counts after fixing:')
print(df['HasCollateral'].value_counts(dropna=False))
print()

HasCollateral value_counts after fixing:
HasCollateral
No     51
Yes    50
NaN     2
Name: count, dtype: int64



In [4]:
# --- PreviousDefaults ---
# Variants: No, Yes, Y, N, 1, 0
defaults_map = {
    'yes': 'Yes', 'y': 'Yes', '1': 'Yes',
    'no': 'No',  'n': 'No',  '0': 'No'
}
df['PreviousDefaults'] = df['PreviousDefaults'].astype(str).str.strip().str.lower().map(
    lambda x: defaults_map.get(x, x.capitalize() if x not in ('nan',) else np.nan)
)
df['PreviousDefaults'] = df['PreviousDefaults'].replace('Nan', np.nan)
print('PreviousDefaults value_counts after fixing:')
print(df['PreviousDefaults'].value_counts(dropna=False))
print()

PreviousDefaults value_counts after fixing:
PreviousDefaults
No     86
Yes    15
NaN     2
Name: count, dtype: int64



In [5]:
# --- Approved ---
# Variants: Yes, No, approved, rejected, Approved, Rejected, YES, NO
approved_map = {
    'yes': 'Yes', 'approved': 'Yes',
    'no': 'No',   'rejected': 'No'
}
df['Approved'] = df['Approved'].astype(str).str.strip().str.lower().map(
    lambda x: approved_map.get(x, np.nan)
)
print('Approved value_counts after fixing:')
print(df['Approved'].value_counts(dropna=False))
print()
print('CHECKPOINT — All category errors fixed.')

Approved value_counts after fixing:
Approved
Yes    68
No     35
Name: count, dtype: int64

CHECKPOINT — All category errors fixed.


## Step 4 — Impute Missing Values

In [6]:
# Numeric columns → median imputation
numeric_cols = ['Income', 'CreditScore', 'EmploymentYears', 'LoanAmount']
for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'  {col}: filled with median = {median_val:.2f}')

# Categorical columns → mode imputation
cat_cols = ['HasCollateral', 'PreviousDefaults', 'Approved']
for col in cat_cols:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f'  {col}: filled with mode = {mode_val}')

print()
print('CHECKPOINT — Missing values after imputation:')
print(df.isnull().sum())

  Income: filled with median = 69460.50
  CreditScore: filled with median = 638.00
  EmploymentYears: filled with median = 12.55
  LoanAmount: filled with median = 20700.00
  HasCollateral: filled with mode = No
  PreviousDefaults: filled with mode = No
  Approved: filled with mode = Yes

CHECKPOINT — Missing values after imputation:
Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


## Step 5 — Remove Duplicates

In [7]:
rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)
print(f'CHECKPOINT — Rows before deduplication: {rows_before}')
print(f'             Rows after  deduplication: {rows_after}')
print(f'             Duplicates removed: {rows_before - rows_after}')

CHECKPOINT — Rows before deduplication: 103
             Rows after  deduplication: 100
             Duplicates removed: 3


## Step 6 — Outlier Capping (IQR Method)

In [8]:
# Cap outliers using IQR with k=1.5 — clip values to [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
# Do NOT delete rows; instead use .clip() to bring extreme values to the fence.

outlier_cols = ['Income', 'CreditScore', 'LoanAmount', 'EmploymentYears']

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before_min, before_max = df[col].min(), df[col].max()
    df[col] = df[col].clip(lower=lower, upper=upper)
    after_min, after_max = df[col].min(), df[col].max()
    print(f'{col}: [{before_min:.1f}, {before_max:.1f}] → capped to [{after_min:.1f}, {after_max:.1f}]')

print()
print('CHECKPOINT — Outliers capped.')

Income: [25851.0, 250000.0] → capped to [25851.0, 167619.1]


CreditScore: [484.0, 920.0] → capped to [484.0, 920.0]
LoanAmount: [5000.0, 180000.0] → capped to [5000.0, 66212.5]
EmploymentYears: [0.5, 25.0] → capped to [0.5, 25.0]

CHECKPOINT — Outliers capped.


## Step 7 — Label Encoding

In [9]:
# Map Yes → 1, No → 0 for binary columns
binary_map = {'Yes': 1, 'No': 0}

for col in ['Approved', 'HasCollateral', 'PreviousDefaults']:
    df[col] = df[col].map(binary_map)
    print(f'{col} encoded. Distribution:')
    print(df[col].value_counts())
    print()

print('CHECKPOINT — Label encoding complete.')

Approved encoded. Distribution:
Approved
1    66
0    34
Name: count, dtype: int64

HasCollateral encoded. Distribution:
HasCollateral
0    52
1    48
Name: count, dtype: int64

PreviousDefaults encoded. Distribution:
PreviousDefaults
0    85
1    15
Name: count, dtype: int64

CHECKPOINT — Label encoding complete.


## Step 8 — Class Balance Check

In [10]:
print('=== CLASS BALANCE — Approved ===')
print('Counts:')
print(df['Approved'].value_counts())
print()
print('Proportions:')
print(df['Approved'].value_counts(normalize=True).round(3))
print()
print('CHECKPOINT — Class balance assessment:')
print('  The dataset is roughly balanced (both classes well-represented).')
print('  Accuracy is therefore a reasonably reliable metric alongside Precision, Recall, and F1.')

=== CLASS BALANCE — Approved ===
Counts:
Approved
1    66
0    34
Name: count, dtype: int64

Proportions:
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64

CHECKPOINT — Class balance assessment:
  The dataset is roughly balanced (both classes well-represented).
  Accuracy is therefore a reasonably reliable metric alongside Precision, Recall, and F1.


## Step 9 — Feature Engineering (No Leakage)

In [11]:
# Build new features from input features ONLY — never touch 'Approved'

# Debt-to-Income ratio: higher ratio → higher financial burden
df['DebtToIncome'] = df['LoanAmount'] / (df['Income'] + 1)  # +1 avoids division by zero

# Income per year of employment: productivity / seniority proxy
df['IncomePerYearEmployed'] = df['Income'] / (df['EmploymentYears'] + 1)

print('CHECKPOINT — Feature engineering complete.')
print('New columns added: DebtToIncome, IncomePerYearEmployed')
print(df[['Income', 'LoanAmount', 'EmploymentYears', 'DebtToIncome', 'IncomePerYearEmployed']].head())

CHECKPOINT — Feature engineering complete.
New columns added: DebtToIncome, IncomePerYearEmployed
     Income  LoanAmount  EmploymentYears  DebtToIncome  IncomePerYearEmployed
0  108810.0     25800.0              1.1      0.237108           51814.285714
1   96482.0     11200.0             15.0      0.116083            6030.125000
2   28478.0     12100.0              5.4      0.424874            4449.687500
3   25851.0      7000.0             17.6      0.270772            1389.838710
4   38396.0     10700.0              9.8      0.278668            3555.185185


## Step 10 — Feature Scaling

### Scaler Choice: RobustScaler

The class script uses `StandardScaler`, which centers data by subtracting the mean and divides by the standard deviation. However, `StandardScaler` is sensitive to outliers — even after IQR capping, financial data like `Income` and `LoanAmount` can still be right-skewed.

I chose **`RobustScaler`** from `sklearn.preprocessing` for this assignment because:
1. It uses the **median** (instead of the mean) and the **interquartile range** (instead of standard deviation) for scaling, making it inherently robust to any remaining extreme values.
2. Loan data is financial in nature and often contains heavy-tailed distributions — `RobustScaler` preserves the distribution shape better than `StandardScaler` in these cases.
3. The IQR-based approach aligns naturally with the IQR capping we already applied in Step 6.

**Source:** Scikit-learn documentation — *RobustScaler*: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html

> **Note:** We do NOT scale the label-encoded binary columns (`Approved`, `HasCollateral`, `PreviousDefaults`) because they are already on a binary 0/1 scale.

In [12]:
from sklearn.preprocessing import RobustScaler

# Only scale continuous numeric features
numeric_features = ['Income', 'CreditScore', 'EmploymentYears', 'LoanAmount',
                    'DebtToIncome', 'IncomePerYearEmployed']

scaler = RobustScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

print('CHECKPOINT — RobustScaler applied to numeric features.')
print('Binary columns (Approved, HasCollateral, PreviousDefaults) left unscaled.')
print()
print(df[numeric_features].describe().round(3))

CHECKPOINT — RobustScaler applied to numeric features.
Binary columns (Approved, HasCollateral, PreviousDefaults) left unscaled.

        Income  CreditScore  EmploymentYears  LoanAmount  DebtToIncome  \
count  100.000      100.000          100.000     100.000       100.000   
mean     0.058        0.085            0.009       0.232         0.032   
std      0.610        0.623            0.599       0.714         0.613   
min     -0.911       -0.997           -1.030      -0.758        -1.232   
25%     -0.449       -0.401           -0.536      -0.304        -0.447   
50%      0.000        0.000            0.000       0.000         0.000   
75%      0.551        0.599            0.464       0.696         0.553   
max      2.051        1.825            1.064       2.196         1.814   

       IncomePerYearEmployed  
count                100.000  
mean                   0.639  
std                    1.814  
min                   -0.718  
25%                   -0.264  
50%              

## Step 11 — Final Checks & Save

In [13]:
print('=== FINAL INFO ===')
df.info()
print()
print('=== FINAL MISSING COUNTS ===')
print(df.isnull().sum())
print()
print('=== FINAL HEAD ===')
print(df.head())
print()
print('=== Approved column (should be 0/1 only) ===')
print(df['Approved'].value_counts())
print()

# Save to CSV
df.to_csv('clean_loan_dataset.csv', index=False)
print('CHECKPOINT — Saved clean_loan_dataset.csv successfully.')
print(f'Final shape: {df.shape}')

=== FINAL INFO ===
<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int64  
 5   PreviousDefaults       100 non-null    int64  
 6   Approved               100 non-null    int64  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.8 KB

=== FINAL MISSING COUNTS ===
Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEm